In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from torch.utils.data import DataLoader
from torchinfo import summary
from src.models import *
from src.helpers import *

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

ACCELS = ['ax', 'ay', 'az', 'atotal']
CLASSES = ['NORMAL', 'POTHOLE']
WINDOW_LENGTHS = {'A': 350, 'B': 250, 'C': 150}
cost_matrix = torch.tensor([[0, 200.0], [400.0, 0]], dtype=torch.float32).to(device)

Using device: mps


In [8]:
df = pd.read_csv('data/intermediary_data/supervised_C3.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nClass distributions:")
print(df.groupby('event_id')['label'].first().value_counts())
df.head()

Shape: (102169, 13)

Columns: ['event_id', 'label', 'test', 'elapsed_sec', 'wall_clock', 'ax', 'ay', 'az', 'atotal', 'lat', 'lon', 'elevation', 'label_time']

Class distributions:
label
NORMAL     533
POTHOLE    147
Name: count, dtype: int64


,event_id,label,test,elapsed_sec,wall_clock,ax,ay,az,atotal,lat,lon,elevation,label_time
0,test_1_NORMAL_224942647231,NORMAL,test_1,22.901,2026-04-19 22:49:41.901000+00:00,0.94,-0.58,0.95,1.46,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
1,test_1_NORMAL_224942647231,NORMAL,test_1,22.911,2026-04-19 22:49:41.911000+00:00,0.73,0.06,0.45,0.86,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
2,test_1_NORMAL_224942647231,NORMAL,test_1,22.921,2026-04-19 22:49:41.921000+00:00,0.77,-0.46,0.63,1.09,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
3,test_1_NORMAL_224942647231,NORMAL,test_1,22.931,2026-04-19 22:49:41.931000+00:00,1.06,-0.52,0.71,1.38,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00
4,test_1_NORMAL_224942647231,NORMAL,test_1,22.941,2026-04-19 22:49:41.941000+00:00,0.84,-0.03,0.70,1.09,44.00799,-73.180151,131.981021,2026-04-19 22:49:42.647231+00:00


In [9]:
events = df.dropna(subset=['lat', 'lon'])
events = events.groupby('event_id').first().reset_index()

# center the map
m = folium.Map(location=[events['lat'].mean(), events['lon'].mean()], 
               zoom_start=14,
               tiles='OpenStreetMap')
# plot normal segments as small blue dots (breadcrumbs)
normal = events[events['label'] == 'NORMAL']
for _, row in normal.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3, # smaller than potholes
        color='blue',
        fill=True,
        fill_opacity=0.4
    ).add_to(m)

# plot potholes as larger red markers
potholes = events[events['label'] == 'POTHOLE']
for _, row in potholes.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup='POTHOLE'
    ).add_to(m)

display(m)